# Homework №2

*Natural Language Processing Course, HSE 2026*

### General Rules and Submission Guidelines

1. Copying code from external sources (**including using LLMs**) without explicit citation is strictly prohibited and will result in 0 points for the entire assignment. If you consult any resources or AI tools, you must clearly state this in a separate Markdown cell. If suspected of this, you might be asked to explain your code to the grader and answer their questions during a separate session to avoid the mentioned penalty.
2. All results must be fully **reproducible**. You are required to use `set_seed` everywhere so that the grader can obtain the same results when rerunning your notebook.
3. The notebook must run from top to bottom without errors. Submissions that fail to execute sequentially will not be accepted.
4. You must satisfy all requirements in each task to receive full credit. Partial completion may lead to partial scoring.
5. Do not modify the original notebook structure or provided Markdown cells. You are only allowed to write code in the sections marked `# TODO: your code here`. Any explanations, interpretations, or additional comments must be placed **in separate Markdown cells**. If you choose to do so, leave an explanation as to what and why was changed.
6. The final submission must be a completed `.ipynb` Jupyter Notebook. You may conduct your experiments in Jupyter Notebook, VS Code, or Google Colab — whichever environment you prefer.

**Note: You may write your textual comments in Markdown for the conducted experiments *in either Russian or English***.

### Environment Setup

Loading necessary libraries. If you need anything else, feel free to add more libraries and dependencies.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)

from datasets import Dataset
import random

# TODO: your code here
import re

from tqdm.auto import tqdm
from datasets import DatasetDict, ClassLabel
from transformers import AutoModelForCausalLM
from sklearn.metrics import accuracy_score

In [ ]:
def set_seed(seed):
    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)

    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(1337)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

### **0. Introduction to the Task and Data**

In this assignment, you will get acquainted with a binary text classification task in Russian, known as Linguistic Acceptability. Your goal will be to train a model capable of distinguishing grammatically correct sentences from incorrect ones.

**RuCoLA (Russian Corpus of Linguistic Acceptability)** is a corpus of Russian sentences annotated with a binary label indicating whether a sentence is linguistically acceptable (grammatically correct and meaningful) or not.

**Structure:** The data is already split into training, validation, and test sets. A detailed description and the dataset itself are available in the project's official [repository](https://github.com/RussianNLP/RuCoLA).

***Important:*** Before starting the assignment, carefully study the repository's `README`. Pay special attention to the sections describing the data structure and the already implemented baselines — this will help you in your work.

### **1. Data Preparation (0.25 point)**



**Data Location:**
The data is located in the [repository](https://github.com/RussianNLP/RuCoLA/tree/main/data). Download the files contained in this folder.




**Files to be Used:**
*   For **training** (`train`), we will use the file **`in_domain_train.csv`**. This file contains examples from linguistic sources.
*   For **testing** (`test`), we will use the file **`in_domain_dev.csv`**.






**Data Format:**
The essential information for us is located in two columns:
*   **`sentence`**: This column contains the text of the sentence (input data for the model).
*   **`acceptable`**: This is the target variable for prediction. It contains a binary label:
    *   **1** — the sentence is linguistically acceptable (grammatically correct).
    *   **0** — the sentence is unacceptable (contains an error).

By the end of this step, you should **have both training and test datasets loaded and ready for further preprocessing** and model experimentation.

In [ ]:
# TODO: your code here
train_url = "https://raw.githubusercontent.com/RussianNLP/RuCoLA/main/data/in_domain_train.csv"
dev_url = "https://raw.githubusercontent.com/RussianNLP/RuCoLA/main/data/in_domain_dev.csv"
df_train = pd.read_csv(train_url)
df_test = pd.read_csv(dev_url)
df_train = df_train[["sentence", "acceptable"]]
df_test = df_test[["sentence", "acceptable"]]

data_train_val = Dataset.from_dict({'text': df_train['sentence'], 'label': df_train['acceptable']})
data_train_val = data_train_val.cast_column("label", ClassLabel(num_classes=2))

train_val_sp = data_train_val.train_test_split(test_size=0.15, shuffle=True, seed=1337, stratify_by_column='label')
data_train = train_val_sp['train']
data_val = train_val_sp['test']


data_test = Dataset.from_dict({'text': df_test['sentence'], 'label': df_test['acceptable']})
data_test = data_test.cast_column("label", ClassLabel(num_classes=2))

data = DatasetDict({
    "train": data_train,
    "val": data_val,
    "test": data_test,
})

data

Casting the dataset:   0%|          | 0/7869 [00:00<?, ? examples/s]

Casting the dataset:   0%|          | 0/983 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 6688
    })
    val: Dataset({
        features: ['text', 'label'],
        num_rows: 1181
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 983
    })
})

**Evaluation Metrics:**
The performance of your models will be evaluated using:

1.  **Accuracy:** The percentage of sentences for which the model correctly predicted the label.
2.  **Matthews Correlation Coefficient (MCC):** This is a more robust metric for binary classification tasks, especially if the classes might be slightly imbalanced. It takes into account all four categories of the confusion matrix (True Positives, True Negatives, False Positives, False Negatives) and returns a value in the range of -1 to +1.

**Important:** You *do not need to implement MCC* from scratch. Import and use the ready-made function from the scikit-learn library:

In [ ]:
from sklearn.metrics import matthews_corrcoef

In [ ]:
# TODO: your code here
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'accuracy': np.mean(preds == labels),
           'mcc': matthews_corrcoef(labels, preds)}

### **2. Baseline: Fine-tuning ruBERT (1 point)**


In this section, you will implement a classifier based on one of the BERT family models adapted for the Russian language. Perform fine-tuning of the model on the training data `in_domain_train.csv` and evaluate its quality on `in_domain_dev.csv`.

**What you need to do:**

1.  **Model Selection (0.25 point)**

For the experiment, you can choose one of the following pre-trained models from the Hugging Face Hub (they differ in size and speed):
*   [`cointegrated/rubert-tiny2`](https://huggingface.co/cointegrated/rubert-tiny2)
*   [`ai-forever/ruBert-base`](https://huggingface.co/ai-forever/ruBert-base)
*   [`ai-forever/ruBert-large`](https://huggingface.co/ai-forever/ruBert-large)



In [ ]:
# TODO: your code here
model_name = 'ai-forever/ruBert-base'
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

collator = DataCollatorWithPadding(tokenizer=tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/590 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/716M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: ai-forever/ruBert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initia

2.  **Data Preparation (0.25 point)**
    
    Convert the texts and labels into a format suitable for `transformers`. Use the tokenizer corresponding to your chosen model.

In [ ]:
# TODO: your code here
data_tokenized = data.map(lambda x: tokenizer(x['text'], truncation=True, padding='max_length',
                                              max_length=256, return_tensors='pt'),
                          batched=True, remove_columns=['text'])


data_tokenized

Map:   0%|          | 0/6688 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/716M [00:00<?, ?B/s]

Map:   0%|          | 0/1181 [00:00<?, ? examples/s]

Map:   0%|          | 0/983 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 6688
    })
    val: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1181
    })
    test: Dataset({
        features: ['label', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 983
    })
})

3. **Fine-tuning (0.25 point)**

Stopping Criterion: Train the model until it reaches a plateau (early stopping by loss). Once the loss stops improving for several epochs (e.g., 2-3), training should be stopped. This helps prevent overfitting.
Record the number of epochs it took to reach the plateau.

In [ ]:
# TODO: your code here
default_args_es = dict(
    num_train_epochs=20,
    save_strategy='epoch',
    eval_strategy='epoch',
    logging_strategy="epoch",
    greater_is_better=False,
    metric_for_best_model='eval_loss',
    per_device_eval_batch_size=16,
    per_device_train_batch_size=16,
    seed=1337,
    data_seed=1337,
    report_to='none',
    save_total_limit=1,

    run_name="BERT-early_stopping",
    load_best_model_at_end=True,
    logging_steps=10,
)


args_es = TrainingArguments(
    output_dir='base_model', # TODO
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0,
    **default_args_es
)


trainer = Trainer(
    model=model,
    args=args_es,
    train_dataset=data_tokenized['train'],
    eval_dataset=data_tokenized['val'],
    data_collator=collator,
    compute_metrics=compute_metrics,
    processing_class=tokenizer,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

trainer.train()
print("stopped at epoch:", trainer.state.epoch)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Epoch,Training Loss,Validation Loss,Accuracy,Mcc
1,0.493501,0.435593,0.814564,0.458082
2,0.293057,0.560390,0.809483,0.440446
3,0.172873,0.675841,0.797629,0.443167


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

stopped at epoch: 3.0


4. **Quality Evaluation (0.25 point)**

After training is complete, make predictions for the entire in_domain_dev.csv file.
Calculate the Accuracy and MCC metrics (using sklearn.metrics.matthews_corrcoef).
Compare the obtained results and comment on them in a Markdown cell.

In [ ]:
# TODO: your code here
trainer.evaluate(data_tokenized['test'])

{'eval_loss': 0.4241320788860321,
 'eval_accuracy': 0.82706002034588,
 'eval_mcc': 0.4958385744406765,
 'eval_runtime': 5.5984,
 'eval_samples_per_second': 175.585,
 'eval_steps_per_second': 11.075,
 'epoch': 3.0}

**Your answer here:** При дообучении модели были получены довольно высокие значения accuracy = 0.83 и MCC = 0.5. Так как это первая из тестируемых моделей, то сравнивать не с чем. Но если брать средний уровень рандомного accuracy = 0.5 и MCC = 0, то наша модель явно лучше , accuracy на 0.33 , MCC на 0.5 соответсвенно

### **3. Zero-shot Approach Using a Generative Model (ruGPT-3) (4.25 point)**


In this section, we will try to solve the task without any training, using only a pre-trained generative language model — **ruGPT-3 Large**.



**Model:** Use [`ai-forever/rugpt3large_based_on_gpt2`](https://huggingface.co/ai-forever/rugpt3large_based_on_gpt2) from the `transformers` library.



**Task:**

1.  **Implement a function to calculate loss for a single text (1.25 point)**

You will need to write a function that:
*   Tokenizes the input text, preparing it for the model (don't forget `return_tensors="pt"` and moving it to the `device`).
*   Passes the tokenized inputs to the model, specifying `labels` (for GPT-2/3 models, labels are usually the same as `input_ids`, since the task is next-token prediction).
*   Extracts the `loss` value from the model's output.
*   Returns the loss value (a number).

    **Important:** To complete this task, you can use [this Colab notebook](https://colab.research.google.com/drive/1BnfhKQQiNAXrKlkIynVPFleQ-epF3I55#scrollTo=fOemj3PbeCZi) and the `transformers` documentation as a reference.



In [ ]:
tokenizer = AutoTokenizer.from_pretrained("ai-forever/rugpt3large_based_on_gpt2")
model = AutoModelForCausalLM.from_pretrained("ai-forever/rugpt3large_based_on_gpt2").to(device)

config.json:   0%|          | 0.00/622 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/574 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/3.14G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.14G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/293 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: ai-forever/rugpt3large_based_on_gpt2
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...23}.attn.masked_bias | UNEXPECTED |  | 
transformer.h.{0...23}.attn.bias        | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [ ]:
def get_loss_num(text):
    '''
    Tokenize the input text and move it to the specified device
    Shift the inputs to create labels for the next-token prediction task
    Move labels to the correct device if you are using GPU
    Calculate loss
    '''
    # TODO: your code here
    inputs = tokenizer(text, truncation=True, padding='max_length',
                       max_length=256, return_tensors='pt').to(device)

    labels = inputs["input_ids"].clone().to(device)

    outputs = model(**inputs, labels=labels)
    loss = outputs.loss

    return loss.item()

2.  **Implement the prediction function `predict_zero_shot(text, pos_template, neg_template)` (1 point)**
    *   The function takes the original text and two templates (prompts) — one for the positive (acceptable) class and one for the negative (unacceptable) class.
    *   The templates should be strings with a placeholder `{}`, where the original text will be inserted. For example: `"This sentence is grammatically correct: {}"`.
    *   Inside the function, calculate the loss for the text inserted into the positive template (`pos_loss`) and the loss for the text inserted into the negative template (`neg_loss`).
    *   Compare the values: if `pos_loss` is **less than** `neg_loss` (the model finds the positive variant more probable), return `1` (the "acceptable" class); otherwise, return `0`.

In [ ]:
# TODO: your code here
def predict_zero_shot(text, pos_template='Корректное предложение: {}', neg_template='Некорректное предложение: {}'):
    pos_loss = get_loss_num(pos_template.format(text))
    neg_loss = get_loss_num(neg_template.format(text))
    if pos_loss < neg_loss:
        return 1
    return 0

3.  **Experiments with prompts and generation parameters (1 point)**

This part consists of two sub-tasks: first, you will find the best prompt pair using a fixed decoding strategy. Then, you will experiment with different generation parameters using that best prompt.

Fix the decoding strategy to greedy search. Use this setting for all prompt evaluations.
Create at least three different pairs of prompts. Vary the wording, punctuation, level of detail, and whether the prompt is a statement, question, or instruction.

For each prompt compute Accuracy and MCC.

Identify the prompt pair that yields the highest performance. This will be your best prompt for the next sub-task.

In [ ]:
# TODO: your code here
# 1
prompt = 'Предложение далее корректное? {}\n'
pos_temp = prompt + 'Да'
neg_temp = prompt + 'Нет'

pred = [predict_zero_shot(text, pos_temp, neg_temp) for text in tqdm(data_test['text'])]

print(f'pos_template: {pos_temp}\nneg_template: {neg_temp}')
print(f'Accuracy: {accuracy_score(data_test['label'], pred):.3f}\nMCC: {matthews_corrcoef(data_test['label'], pred):.3f}\n')


#2
prompt = 'Ты лингвист, всю жизнь изучаешь русский язык. Правильно ли, что следующее предложение корректно? {}\n'
pos_temp = prompt + 'Предложение корректно'
neg_temp = prompt + 'Предложение некорректно'

pred = [predict_zero_shot(text, pos_temp, neg_temp) for text in tqdm(data_test['text'])]

print(f'pos_template: {pos_temp}\nneg_template: {neg_temp}')
print(f'Accuracy: {accuracy_score(data_test['label'], pred):.3f}\nMCC: {matthews_corrcoef(data_test['label'], pred):.3f}\n')


# 3
prompt = 'Помоги моей пожилой бабушке, она написала предложение {}, оно корректное?\n'
pos_temp = prompt + 'Да'
neg_temp = prompt + 'Нет'

pred = [predict_zero_shot(text, pos_temp, neg_temp) for text in tqdm(data_test['text'])]

print(f'pos_template: {pos_temp}\nneg_template: {neg_temp}')
print(f'Accuracy: {accuracy_score(data_test['label'], pred):.3f}\nMCC: {matthews_corrcoef(data_test['label'], pred):.3f}\n')

# 4
system_prompt = 'Предложение {} содержит ошибки?\n'
pos_temp = system_prompt + 'Нет'
neg_temp = system_prompt + 'Да'

pred = [predict_zero_shot(text, pos_temp, neg_temp) for text in tqdm(data_test['text'])]

print(f'pos_template: {pos_temp}\nneg_template: {neg_temp}')
print(f'Accuracy: {accuracy_score(data_test['label'], pred):.3f}\nMCC: {matthews_corrcoef(data_test['label'], pred):.3f}\n')

  0%|          | 0/983 [00:00<?, ?it/s]

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


pos_template: Предложение далее корректное? {}
Да
neg_template: Предложение далее корректное? {}
Нет
Accuracy: 0.745
MCC: -0.019



  0%|          | 0/983 [00:00<?, ?it/s]

pos_template: Ты лингвист, всю жизнь изучаешь русский язык. Правильно ли, что следующее предложение корректно? {}
Предложение корректно
neg_template: Ты лингвист, всю жизнь изучаешь русский язык. Правильно ли, что следующее предложение корректно? {}
Предложение некорректно
Accuracy: 0.729
MCC: 0.114



  0%|          | 0/983 [00:00<?, ?it/s]

pos_template: Помоги моей пожилой бабушке, она написала предложение {}, оно корректное?
Да
neg_template: Помоги моей пожилой бабушке, она написала предложение {}, оно корректное?
Нет
Accuracy: 0.712
MCC: -0.042



  0%|          | 0/983 [00:00<?, ?it/s]

pos_template: Предложение {} содержит ошибки?
Нет
neg_template: Предложение {} содержит ошибки?
Да
Accuracy: 0.340
MCC: 0.058



Using the best prompt from previous step, experiment with at least three different generation strategies. The loss calculation can be affected by how the model processes the input. You can modify the generation parameters even though you are only computing loss

For each configuration, evaluate on the test set and record Accuracy and MCC

In [ ]:
# TODO: your code here
def get_loss_num_for_generation_strategies(text, generation_args):
    if generation_args is None:
        generation_args = {}
    max_len = generation_args.get("max_length", 256)
    trunc_side = generation_args.get("truncation_side", "right")
    old_side = getattr(tokenizer, "truncation_side", "right")
    tokenizer.truncation_side = trunc_side
    inputs = tokenizer(text, truncation=True, max_length=max_len, padding="max_length", return_tensors="pt")
    tokenizer.truncation_side = old_side
    inputs = {k: v.to(device) for k, v in inputs.items()}
    labels = inputs["input_ids"].clone()
    with torch.no_grad():
        outputs = model(**inputs, labels=labels)
    return outputs.loss.item()

In [ ]:
# TODO: your code here
def predict_zero_shot_for_generation_strategies(text, pos_template='Корректное предложение: {}', neg_template='Некорректное предложение: {}', generation_args=None):
    pos_loss = get_loss_num_for_generation_strategies(pos_template.format(text), generation_args)
    neg_loss = get_loss_num_for_generation_strategies(neg_template.format(text), generation_args)
    if pos_loss < neg_loss:
        return 1
    return 0

In [ ]:

# По MCC = 0.114 эта пара лучшая
prompt = 'Ты лингвист, всю жизнь изучаешь русский язык. Правильно ли, что следующее предложение корректно? {}\n'
pos_temp = prompt + 'Предложение корректно'
neg_temp = prompt + 'Предложение некорректно'


# 1 полный контекст, вся длина
generation_args = {"max_length": 256, "truncation_side": "right"}

pred = [predict_zero_shot_for_generation_strategies(text, pos_temp, neg_temp, generation_args) for text in tqdm(data_test['text'])]

print(f'generation_args: {generation_args}')
print(f'Accuracy: {accuracy_score(data_test['label'], pred):.3f}\nMCC: {matthews_corrcoef(data_test['label'], pred):.3f}\n')

# 2 обрезка справа до 128 токенов
generation_args = {"max_length": 128, "truncation_side": "right"}

pred = [predict_zero_shot_for_generation_strategies(text, pos_temp, neg_temp, generation_args) for text in tqdm(data_test['text'])]

print(f'generation_args: {generation_args}')
print(f'Accuracy: {accuracy_score(data_test['label'], pred):.3f}\nMCC: {matthews_corrcoef(data_test['label'], pred):.3f}\n')

# 3 обрезка слева до 128 токенов
generation_args = {"max_length": 128, "truncation_side": "left"}

pred = [predict_zero_shot_for_generation_strategies(text, pos_temp, neg_temp, generation_args) for text in tqdm(data_test['text'])]

print(f'generation_args: {generation_args}')
print(f'Accuracy: {accuracy_score(data_test['label'], pred):.3f}\nMCC: {matthews_corrcoef(data_test['label'], pred):.3f}\n')

# 4: обрезка до 64 токенов
generation_args = {"max_length": 64, "truncation_side": "right"}

pred = [predict_zero_shot_for_generation_strategies(text, pos_temp, neg_temp, generation_args) for text in tqdm(data_test['text'])]

print(f'generation_args: {generation_args}')
print(f'Accuracy: {accuracy_score(data_test["label"], pred):.3f}\nMCC: {matthews_corrcoef(data_test["label"], pred):.3f}\n')

  0%|          | 0/983 [00:00<?, ?it/s]

generation_args: {'max_length': 256, 'truncation_side': 'right'}
Accuracy: 0.729
MCC: 0.114



  0%|          | 0/983 [00:00<?, ?it/s]

generation_args: {'max_length': 128, 'truncation_side': 'right'}
Accuracy: 0.579
MCC: -0.128



  0%|          | 0/983 [00:00<?, ?it/s]

generation_args: {'max_length': 128, 'truncation_side': 'left'}
Accuracy: 0.579
MCC: -0.128



4.  **Analysis of Results (1 point)**
    *   Compare the metrics obtained with different prompts.
    *   Draw conclusions about how the following factors influence quality:
        *   The wording of the task (statement, question, instruction).
        *   The presence of punctuation (a period at the end, a question mark).
        *   The length and specificity of the prompt.
    *   Try to explain why one pair of prompts worked better than another.

**Your answer here:** По полученным результатам можно сделать вывод о том, что промпты важны, так как от них зависит результат генерации модели, и следовательно, результат классификации. В зависимости от промптов метрики изменялись следующим образом: accuracy 0.340-0.745 и MCC (-0.042)-0.114. Лучший MCC показал промпт который содержит в себе четкие инструкци : "Ты лингвист, всю жизнь изучаешь русский язык." и соответсвенно продолжение об оценке корректности предложения . Худший accuracy = 0.340 у промпта о содержании ошибки в тексте, скорее всего модель не понимала, о каких именно ошибках идеть речь и отсутсвия четких указаний. От стратегии генерации тоже зависит результат классификации, но незначительно. При обрезке до 128 токенов слева или справа, качесво падает accuracy = 0.579, MCC = -0.128, при полном контексте метрики выше accuracy = 0.729, MCC = 0.114, то есть длина контекста влияет на результат классификации

### **4. Instruction-Tuned Model: Qwen2.5-Instruct (2 points)**



In this section, you will work with the **Qwen2.5-Instruct** model. Your goal is to select prompts that will allow the model to solve the binary classification task of linguistic acceptability without any fine-tuning, relying solely on its ability to follow instructions.



**Model:**
*   Use the base instruction-tuned version: [`Qwen/Qwen2.5-1.5B-Instruct`](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct).
*   *Note:* You may also experiment with larger versions (e.g., `Qwen/Qwen2.5-7B-Instruct`) and compare the results.


**Task:**

1.  **Design a prompt for the task (0.5 point)**

Create at least three different prompts that set various contexts, roles, or behavioral constraints for the model and ask the model to classify a sentence as acceptable (1) or unacceptable (0). The prompt should be passed to the model together with the input sentence.

**!Do not use a system prompt!** in this part - only a single user message or a direct instruction.


In [ ]:
# TODO: your code here

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B-Instruct").to(device)

batch_size = 64

prompts = [
    'Предложение "{}" корректное? Ответь да (1) или нет (0).\nОтвет:',
    'Help my elderly grandmother determine the correctness of the sentence "{}". If the sentence is correct, then answer 1 otherwise 0.\nAnswer:',
    'Пример: «Моя мама любит хризантемы.» → 1 (корректно). Оцени предложение «{}»: 0 или 1?\nОтвет:'
]

answers = [[] for i in range(len(prompts))]

for prompt_num, prompt in tqdm(enumerate(prompts)):
    prompted_texts = [prompt.format(text) for text in data_test['text']]

    for i in tqdm(range(0, len(prompted_texts)//batch_size+1)):
        prompted_texts_batch = prompted_texts[i*batch_size:(i+1)*batch_size]


        inputs = tokenizer(prompted_texts_batch, truncation=True, padding='max_length',
                            max_length=256, return_tensors='pt', add_special_tokens=True,
                            padding_side='left').to(device)

        outputs = model.generate(**inputs, num_beams=1, do_sample=False, max_new_tokens=3)
        input_len = inputs['input_ids'].shape[1]
        generated_only = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
        for g in generated_only:
            answers[prompt_num].append(g)

    print(answers[prompt_num][:5])

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

[' 1\n\n', ' 1\n\n', ' 1\n\n', ' 1\n\n', ' 1\n\n']


  0%|          | 0/16 [00:00<?, ?it/s]

[' 0\n', ' 1\n', ' 1\n', ' 1\n', ' 1\n\n']


  0%|          | 0/16 [00:00<?, ?it/s]

[' 0\n\n', ' 1\n\n', ' 0\n\n', ' 0\n\n', ' 0\n\n']


2.  **Implement response post-processing (0.5 point)**

The model generates free text, so you need to extract the predicted label (0 or 1) from its output. Write a function that:
    *   Takes the generated model text as input.
    *   Extracts the integer 0 or 1 from it. Handle cases where the model provides additional explanations (e.g., look for the first occurrence of "1" or "0", use simple regular expressions).
    *   If the response is ambiguous, decide on a processing strategy. Describe your approach.


 Ищем в тексте первое вхождение цифры 0 или 1, если найдено, то возвращаем её как целое число. Если модель добавила пояснения, например «1 предложение корректно», то учитывается только первая цифра. Если ни 0, ни 1 не найдены, то есть ответ пустой или неоднозначный, возвращаем 0 по умолчанию.

In [ ]:
# TODO: your code here
def post_process(text):
    pattern = r'[01]'
    result = re.findall(pattern, text)
    if len(result):
        return int(result[0])

    return 0

preds = [[] for i in range(len(answers))]

for answer_num, answer in enumerate(answers):
    preds[answer_num] = [post_process(text) for text in answers[answer_num]]

3.  **Experiments with prompts (0.5 point)**

Conduct experiments by varying prompt characteristics. For each variant, evaluate the model on the entire `in_domain_dev.csv` file and compute **Accuracy** and **MCC**. Consider at least the following dimensions:



  *   **Prompt Language:** prompts in Russian vs. prompts in English. How does language proficiency affect the result?
  *   **Instruction Style:** compare different formulations.
  *   **Prompt Length and Detail:** short vs. detailed instructions; you can try adding one example (few-shot).




In [ ]:
# TODO: your code here

for i in range(len(preds)):
    print(f'Prompt №{i}:\nAccuracy: {accuracy_score(data_test['label'], preds[i]):.3f}\nMCC: {matthews_corrcoef(data_test['label'], preds[i]):.3f}\n\n')

Prompt №0:
Accuracy: 0.615
MCC: -0.109


Prompt №1:
Accuracy: 0.745
MCC: -0.019


Prompt №2:
Accuracy: 0.703
MCC: -0.040


Prompt №3:
Accuracy: 0.256
MCC: -0.111





4.  **Analysis of Results (0.5 point)**
    *   Compare the quality of different prompts. Which ones work better and why?
    *   Draw conclusions about the influence of the instruction language.
    *   Provide your thoughts on whether the model size is sufficient to solve this task with good quality.
    *   Describe your findings in MarkDown.

**Your answer here:** При сравнении значений метрик классификации в зависимости от промпта было выявлено, что лучшим промптом из заявленных является 'Предложение "{}" корректное? Ответь да (1) или нет (0).\nОтвет:'.

Модель Qwen2.5-1.5B-Instruct показывает результаты чуть хуже, чем rugpt3large_based_on_gpt2.

Так как значения метрик получились не самые высокие, то можно предположить, что размер модели недостаточен.

Пока что лучшие значения метрик получены дообученной ruBert-base, может нужно выбрать другую стратегию или изменить промпты.

**Your answer here:** При сравнении четырёх промптов лучший результат дал **№1** (короткий вопрос на русском): Accuracy 0.745, MCC −0.019. Хуже всего сработал **№3** (с одним примером, few-shot): Accuracy 0.256, MCC −0.111 — модель, по всей видимости, плохо следует формату «пример → ответ, затем оцени следующее». Промпты №0 (инструкция, RU) и №2 (просьба, EN) заняли промежуточное положение (0.615 и 0.703 по accuracy).

**Влияние языка инструкции:** русскоязычный вопрос (№1) оказался лучше английского промпта (№2): 0.745 против 0.703. Для задачи на русской приемлемости формулировка на русском дала более стабильный результат.

**Размер модели:** метрики Qwen2.5-1.5B-Instruct ниже, чем у ruGPT-3 и заметно ниже, чем у дообученной ruBert-base, поэтому для данной задачи без дообучения размера 1.5B может быть недостаточно; качество сильно зависит от формулировки промпта.

### **5. Adding a System Prompt (0,75 point)**


Add a System Prompt and **conduct an experiment similar to** **part 4**, varying the System Prompt. Draw conclusions about the influence of the System Prompt. Create **at least three different system prompts** that set various contexts, roles, or behavioral constraints for the model.

In [ ]:
# TODO: your code here
user_prompts = [
    'Предложение "{}" корректное? Ответь да (1) или нет (0).\nОтвет:',
    'Help my elderly grandmother determine the correctness of the sentence "{}". If the sentence is correct, then answer 1 otherwise 0.\nAnswer:',
    'Пример: «Моя мама любит хризантемы.» → 1 (корректно). Оцени предложение «{}»: 0 или 1?\nОтвет:'
]

system_prompts = [
    'Ты отличный лингвист, всю жизнь изучаешь русский язык. Твоя задача оценивать грамматическую корректность. ',
    'You are a strict grammar checker. Output only 0 or 1, no explanations. ',
    'Ты эксперт-редактор: твоя единственная задача ставить 0 (ошибка) или 1 (всё верно). Никаких пояснений. '
]

prompts = [sys_p + user_p for sys_p in system_prompts for user_p in user_prompts]

answers = [[] for i in range(len(prompts))]

for prompt_num, prompt in tqdm(enumerate(prompts)):
    prompted_texts = [prompt.format(text) for text in data_test['text']]

    for i in tqdm(range(0, len(prompted_texts)//batch_size+1)):
        prompted_texts_batch = prompted_texts[i*batch_size:(i+1)*batch_size]

        inputs = tokenizer(prompted_texts_batch, truncation=True, padding='max_length',
                            max_length=256, return_tensors='pt', add_special_tokens=True,
                            padding_side='left').to(device)

        outputs = model.generate(**inputs, num_beams=1, do_sample=False, max_new_tokens=3)
        input_len = inputs['input_ids'].shape[1]
        generated_only = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
        for g in generated_only:
            answers[prompt_num].append(g)

    print(answers[prompt_num][:5])


preds = [[] for i in range(len(answers))]

for answer_num, answer in enumerate(answers):
    preds[answer_num] = [post_process(text) for text in answers[answer_num]]


for i in range(len(preds)):
    sys_num, user_num = i // len(user_prompts), i % len(user_prompts)
    print(f'System №{sys_num}, User №{user_num}:\nAccuracy: {accuracy_score(data_test["label"], preds[i]):.3f}\nMCC: {matthews_corrcoef(data_test["label"], preds[i]):.3f}\n\n')

0it [00:00, ?it/s]

  0%|          | 0/16 [00:00<?, ?it/s]

[' 0\n', ' 1\n', ' 1\n', ' 1\n', ' 0\n']


  0%|          | 0/16 [00:00<?, ?it/s]

[' 1\n\n', ' 1\n\n', ' 1\n\n', ' 1\n\n', ' 1\n\n']


...

[' 0\n\n', ' 1\n\n', ' 1\n\n', ' 1\n\n', ' 0\n\n']


  0%|          | 0/16 [00:00<?, ?it/s]

[' 0\n', ' 1\n', ' 1\n', ' 1\n', ' 0\n']


  0%|          | 0/16 [00:00<?, ?it/s]

[' 0\n\n', ' 0\n\n', ' 1\n\n', ' 0\n\n', ' 0\n\n']
System №0, User №0:
Accuracy: 0.458
MCC: -0.006


System №0, User №1:
Accuracy: 0.726
MCC: 0.012


System №0, User №2:
Accuracy: 0.715
MCC: 0.002


System №0, User №3:
Accuracy: 0.361
MCC: -0.047


System №1, User №0:
Accuracy: 0.651
MCC: 0.025


System №1, User №1:
Accuracy: 0.745
MCC: 0.010


System №1, User №2:
Accuracy: 0.707
MCC: 0.021


System №1, User №3:
Accuracy: 0.704
MCC: 0.037


System №2, User №0:
Accuracy: 0.253
MCC: -0.055


System №2, User №1:
Accuracy: 0.536
MCC: -0.015


System №2, User №2:
Accuracy: 0.454
MCC: -0.059


System №2, User №3:
Accuracy: 0.388
MCC: 0.043




**Your answer here:** Системный промпт изменил предсказательную способность модели, а именно:

* accuracy: у одного промпта увеличилось, у одного уменьшилось, а у одного неизменилось.
* mcc: у двух промптов увеличилось, а у одного уменьшилось.

У самого лучшего промпта (под номером 1) accuracy не изменилось, а mcc чуть выросло, это можно интерпретировать как неухудшение предсказательной способности модели.

**Your answer here:** Системный промпт заметно влияет на предсказательную способность модели.

При одном и том же пользовательском промпте (User №1, вопрос «Предложение корректное?») результаты по системным промптам различаются: **System №0** (лингвист, RU) — Accuracy 0.726; **System №1** (strict grammar checker, EN) — 0.745; **System №2** (эксперт-редактор, RU) — 0.536. Лучший результат даёт системный промпт №1 (короткая роль на английском с жёстким ограничением «Output only 0 or 1»). Системный промпт №2 сильно ухудшает качество по сравнению с №0 и №1.

По комбинациям: наивысшая Accuracy (0.745) — System №1, User №1; лучший MCC среди всех (0.043) — System №2, User №3. То есть добавление системного промпта по-разному взаимодействует с пользовательским: для одних пар метрики растут, для других падают. Вывод: формулировка и язык системного промпта важны; слишком жёсткая или неочевидная роль (как у №2) может ухудшить результат по сравнению с нейтральным или чётким системным промптом (№0, №1).

### **6. Base (Non-Instruction) Version of Qwen2.5 (0,75 point)**


Conduct an experiment similar to the approach in section 4-5, but using the base non-instruction version of the **Qwen/Qwen2.5-1.5B** model. Use [`Qwen/Qwen2.5-1.5B`](https://huggingface.co/Qwen/Qwen2.5-1.5B) (or `Qwen/Qwen2.5-7B`).

Use exactly the same hyperparameters, generation strategy, and system prompts that turned out to be optimal for the instruction-tuned version.

In [ ]:
# TODO: your code here
# Базовая модель (без instruct). Те же гиперпараметры и стратегия, что в 4–5. Оптимальный системный промпт из части 5 — System №1 (EN grammar checker).

tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-1.5B")
model = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-1.5B").to(device)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

batch_size = 64
system_prompt = 'You are a strict grammar checker. Output only 0 or 1, no explanations. '

prompts = [
    'Дано предложение "{}"\nКлассифицируй это предложение на корректное (1) и некорректное (0).\nОтвет:',
    'Предложение "{}" корректное? Ответь да (1) или нет (0).\nОтвет:',
    'Help my elderly grandmother determine the correctness of the sentence "{}". If the sentence is correct, then answer 1 otherwise 0.\nAnswer:',
    system_prompt + 'Дано предложение "{}"\nКлассифицируй это предложение на корректное (1) и некорректное (0).\nОтвет:',
    system_prompt + 'Предложение "{}" корректное? Ответь да (1) или нет (0).\nОтвет:',
    system_prompt + 'Help my elderly grandmother determine the correctness of the sentence "{}". If the sentence is correct, then answer 1 otherwise 0.\nAnswer:'
]

answers = [[] for i in range(len(prompts))]

for prompt_num, prompt in tqdm(enumerate(prompts)):
    prompted_texts = [prompt.format(text) for text in data_test['text']]

    for i in tqdm(range(0, len(prompted_texts)//batch_size+1)):
        prompted_texts_batch = prompted_texts[i*batch_size:(i+1)*batch_size]

        inputs = tokenizer(prompted_texts_batch, truncation=True, padding='max_length',
                            max_length=256, return_tensors='pt', add_special_tokens=True,
                            padding_side='left').to(device)

        outputs = model.generate(**inputs, num_beams=1, do_sample=False, max_new_tokens=3)
        input_len = inputs['input_ids'].shape[1]
        generated_only = tokenizer.batch_decode(outputs[:, input_len:], skip_special_tokens=True)
        for g in generated_only:
            answers[prompt_num].append(g)

    print(answers[prompt_num][:5])

preds = [[] for i in range(len(answers))]

for answer_num, answer in enumerate(answers):
    preds[answer_num] = [post_process(text) for text in answers[answer_num]]


for i in range(len(preds)):
    print(f'Prompt №{i}:\nAccuracy: {accuracy_score(data_test["label"], preds[i]):.3f}\nMCC: {matthews_corrcoef(data_test["label"], preds[i]):.3f}\n\n')

config.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

[' 0', ' 1', ' 1', ' 1', ' 1']


  0%|          | 0/16 [00:00<?, ?it/s]

Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:151643 for

[' 0', ' 0', ' 0', ' 0', ' 0']
Prompt №0:
Accuracy: 0.459
MCC: -0.067


Prompt №1:
Accuracy: 0.468
MCC: 0.018


Prompt №2:
Accuracy: 0.304
MCC: -0.050


Prompt №3:
Accuracy: 0.383
MCC: 0.044


Prompt №4:
Accuracy: 0.663
MCC: -0.046


Prompt №5:
Accuracy: 0.264
MCC: 0.020




**Your answer here:** При использовании того же самого кода для модели, которая обучалась без применения инструкций, для ответов на некоторые вопросы модель вместо 1 и 0 стала выдавать да и нет (иногда вперемешку с 0 и 1), что привело к ошибкам при подсчете метрик для этих промптов.

Для большинства промптов (в том числе с системными промптами) модель Qwen2.5-1.5B показала результаты хуже, чем Qwen2.5-1.5B-Instruct. Это скорее всего связано с особенностью обучения (так как Qwen2.5-1.5B-Instruct дообучалась на наборе с инструкциями).

**Your answer here:** При использовании того же самого кода для модели, которая обучалась без применения инструкций, для ответов на некоторые вопросы модель вместо 1 и 0 стала выдавать да и нет (иногда вперемешку с 0 и 1), что привело к ошибкам при подсчете метрик для этих промптов.

По результатам: лучшая комбинация у базовой модели — Prompt №4 (системный + вопрос «Предложение корректное?»): Accuracy 0.663, MCC −0.046. Это ниже, чем у Qwen2.5-1.5B-Instruct с той же парой (0.745 / 0.010). Для большинства промптов (в том числе с системными) модель Qwen2.5-1.5B показала результаты хуже, чем Qwen2.5-1.5B-Instruct. Это скорее всего связано с особенностью обучения (Qwen2.5-1.5B-Instruct дообучалась на наборе с инструкциями).

### **7. Draw conclusions (1 point)**


Describe whether the results of the instruction-tuned and non-instruction versions of the same base model differ. Explain what these differences (or lack thereof) are related to. How does instruction fine-tuning affect the model's ability to solve the linguistic acceptability task?

**Your answer here:** В условиях проведенных экспериментов лучшие значения метрик наблюдались у модели ruBert-base, у моделей rugpt3large_based_on_gpt2, Qwen2.5-1.5B-Instruct и Qwen2.5-1.5B значения хуже и сильно зависят от промптов (от стратегий генерации и от наличия системного промпта почти не зависят). Это может быть связано с архитектурными особенностями и особенностями обучения моделей (ruBert-base предсказывает сразу класс, а остальные генерируют продолжение запроса). Поэтому при необходимости решения узконаправленных задач следует сначала проводить эксперименты на более узконаправленных моделях.

**Your answer here:** Результаты instruction-tuned (Qwen2.5-1.5B-Instruct) и non-instruction (Qwen2.5-1.5B) версий одной и той же базовой модели различаются: у Instruct лучшие метрики (например, Accuracy до 0.745 при удачной паре промптов), у базовой версии лучший результат заметно ниже (0.663) и модель чаще выдаёт «да/нет» вместо 0/1, что искажает подсчёт. Различия связаны с дообучением на инструкциях: Instruct приучена следовать формату ответа и формулировке задачи, базовая — только к продолжению текста. Instruction fine-tuning улучшает способность модели решать задачу лингвистической приемлемости за счёт явного обучения следовать инструкциям и выдавать ответ в нужном формате.

В условиях проведенных экспериментов лучшие значения метрик наблюдались у модели ruBert-base; у моделей rugpt3large_based_on_gpt2, Qwen2.5-1.5B-Instruct и Qwen2.5-1.5B значения хуже и сильно зависят от промптов (от стратегий генерации и от наличия системного промпта почти не зависят). Это может быть связано с архитектурными особенностями и особенностями обучения моделей (ruBert-base предсказывает сразу класс, а остальные генерируют продолжение запроса). Поэтому при необходимости решения узконаправленных задач следует сначала проводить эксперименты на более узконаправленных моделях.